[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mento-calc/mento/blob/main/docs/source/examples/beam_summary_ACI_318-19%20Design.ipynb)

In [1]:
# Google Colab does not ship mento, so install it when running there.
# Guarded so that building the documentation never shells out to pip.
import sys

if "google.colab" in sys.modules:
    %pip install mento

# Beam summary design with ACI 318-19

**Create input dataframe**

This example builds the two tables, sections and forces, directly in the cell; it is easier to read them from the sheets `Sections` and `Forces` of an Excel file with `from_excel`, as the User Guide explains.

The sections table has no reinforcement, since mento designs it.

In [2]:
from mento import MPa
from mento import BeamSummary, Concrete_ACI_318_19, SteelBar
import pandas as pd

conc = Concrete_ACI_318_19(name="C25", f_c=25 * MPa)
steel = SteelBar(name="ADN 420", f_y=420 * MPa)

# One row per section: its geometry.
# Reinforcement columns left out are zero: this input has no reinforcement, mento designs it.
sections = pd.DataFrame(
    {
        "Label": ["", "V101", "V102", "V103", "V104"],
        "b": ["cm", 20, 20, 20, 20],
        "h": ["cm", 50, 50, 50, 50],
        "cc": ["mm", 25, 25, 25, 25],
    }
)
# One row per load combination of a section: Nx > 0 compression, My > 0 bottom in tension.
forces = pd.DataFrame(
    {
        "Label": ["", "V101", "V102", "V103", "V104"],
        "Comb.": ["", "ELU 1", "ELU 2", "ELU 3", "ELU 4"],
        "Nx": ["kN", 0, 0, 0, 0],
        "Vz": ["kN", 20, -50, 100, 100],
        "My": ["kNm", 0, 35, 80, -80],
    }
)

beam_summary = BeamSummary(conc, steel, sections, forces)
# The sections table, as beam_summary.to_excel() writes it
beam_summary.sections_table

,Label,b,h,cc,legs,dbs,sl,n1_top,db1_top,n2_top,...,n4_top,db4_top,n1_bot,db1_bot,n2_bot,db2_bot,n3_bot,db3_bot,n4_bot,db4_bot
0,,cm,cm,mm,,mm,cm,,mm,,...,,mm,,mm,,mm,,mm,,mm
1,V101,20,50,25,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,V102,20,50,25,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,V103,20,50,25,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,V104,20,50,25,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


**Design all beams for shear and flexure**

In [3]:
beam_summary.design()

✅ Design completed for every section of the summary.


,Label,b,h,cc,legs,dbs,sl,n1_top,db1_top,n2_top,...,n4_top,db4_top,n1_bot,db1_bot,n2_bot,db2_bot,n3_bot,db3_bot,n4_bot,db4_bot
0,,cm,cm,mm,,mm,cm,,mm,,...,,mm,,mm,,mm,,mm,,mm
1,V101,20,50,25,2,10,22,0,0,0,...,0,0,2,12,0,0,0,0,0,0
2,V102,20,50,25,2,10,22,0,0,0,...,0,0,2,12,1,10,0,0,0,0
3,V103,20,50,25,2,10,22,0,0,0,...,0,0,2,16,1,12,0,0,0,0
4,V104,20,50,25,2,10,22,2,16,1,...,0,0,2,12,0,0,0,0,0,0


**Check all beams for shear and flexure with current design**

In [4]:
beam_summary.check()

,Beam,b×h,"As,top","As,bot",Av,"Comb.,top","Mu,top","DCRb,top","Comb.,bot","Mu,bot","DCRb,bot","Comb.,v",Vu,Nu,DCRv,Warnings,Ok?
0,,cm,,,,,kNm,,,kNm,,,kN,kN,,,
1,V101,20×50,-,2Ø12,1sØ10/22,-,NaN,0.0,-,NaN,0.0,ELU 1,20.0,0.0,0.124,-,✅
2,V102,20×50,-,2Ø12+1Ø10,1sØ10/22,-,NaN,0.0,ELU 2,35.0,0.684,ELU 2,50.0,0.0,0.309,-,✅
3,V103,20×50,-,2Ø16+1Ø12,1sØ10/22,-,NaN,0.0,ELU 3,80.0,0.951,ELU 3,100.0,0.0,0.62,-,✅
4,V104,20×50,2Ø16+1Ø12,2Ø12,1sØ10/22,ELU 4,-80.0,0.951,-,NaN,0.0,ELU 4,100.0,0.0,0.62,-,✅


**Export design results to excel**

In [5]:
beam_summary.export_design("Beam-Design.xlsx")

✅ Sections and forces written to Beam-Design.xlsx


At this point you could open the Excel file, edit the bars or the stirrups in its `Sections` sheet (both faces are written), import it and check again.

In [6]:
beam_summary.import_design("Beam-Design.xlsx")

✅ Sections and forces read from Beam-Design.xlsx


The sections table, as read back from the file.

In [7]:
beam_summary.sections_table

,Label,b,h,cc,legs,dbs,sl,n1_top,db1_top,n2_top,...,n4_top,db4_top,n1_bot,db1_bot,n2_bot,db2_bot,n3_bot,db3_bot,n4_bot,db4_bot
0,,cm,cm,mm,,mm,cm,,mm,,...,,mm,,mm,,mm,,mm,,mm
1,V101,20,50,25,2,10,22,0,0,0,...,0,0,2,12,0,0,0,0,0,0
2,V102,20,50,25,2,10,22,0,0,0,...,0,0,2,12,1,10,0,0,0,0
3,V103,20,50,25,2,10,22,0,0,0,...,0,0,2,16,1,12,0,0,0,0
4,V104,20,50,25,2,10,22,2,16,1,...,0,0,2,12,0,0,0,0,0,0


Perform the check again, with the new rebar.

In [8]:
beam_summary.check()

,Beam,b×h,"As,top","As,bot",Av,"Comb.,top","Mu,top","DCRb,top","Comb.,bot","Mu,bot","DCRb,bot","Comb.,v",Vu,Nu,DCRv,Warnings,Ok?
0,,cm,,,,,kNm,,,kNm,,,kN,kN,,,
1,V101,20×50,-,2Ø12,1sØ10/22,-,NaN,0.0,-,NaN,0.0,ELU 1,20.0,0.0,0.124,-,✅
2,V102,20×50,-,2Ø12+1Ø10,1sØ10/22,-,NaN,0.0,ELU 2,35.0,0.684,ELU 2,50.0,0.0,0.309,-,✅
3,V103,20×50,-,2Ø16+1Ø12,1sØ10/22,-,NaN,0.0,ELU 3,80.0,0.951,ELU 3,100.0,0.0,0.62,-,✅
4,V104,20×50,2Ø16+1Ø12,2Ø12,1sØ10/22,ELU 4,-80.0,0.951,-,NaN,0.0,ELU 4,100.0,0.0,0.62,-,✅


You could check the specific shear and flexure results following the Beam Summary Check example.

**Export results to Word**

In [9]:
beam_summary.results_detailed_doc(index=1)

✅ Results exported to Beam_Summary_ACI 318-19.docx


**Export results to Excel**

In [10]:
beam_summary.check().to_excel("Beam_summary_results.xlsx", index=False)